In [ ]:
import numpy as np
import anndata
import dynamo as dyn
import matplotlib.pyplot as plt


# -------------------------------------------------
# CHOOSE DATASET HERE
# -------------------------------------------------


# -----------------------------
# DATASET REGISTRY
# -----------------------------
DATASETS = {
    "cell_cycle": {
        "X": "./data/real_data_benchmark/cell_cycle/X_cc.npy",
        "V": "./data/real_data_benchmark/cell_cycle/V_cc.npy",
        "color": "./data/real_data_benchmark/cell_cycle/color_cell_cycle_relativePos.npy",
        "embedding": "./data/real_data_benchmark/cell_cycle/flowmap_embedding.npy",
    },
    "pancreas": {
        "X": "./data/real_data_benchmark/pancreas/X_pca.npy",
        "V": "./data/real_data_benchmark/pancreas/V_pca_stochastic.npy",
        "color": "./data/real_data_benchmark/pancreas/pseudotime.npy",
        "embedding": "./data/real_data_benchmark/pancreas/scvelo_umap_embedding.npy",
    },
    "dentate_gyrus": {
        "X": "./data/real_data_benchmark/dentate_gyrus/X_pca.npy",
        "V": "./data/real_data_benchmark/dentate_gyrus/V_pca_dynamical.npy",
        "color": "./data/real_data_benchmark/dentate_gyrus/pseudotime.npy",
        "embedding": "./data/real_data_benchmark/dentate_gyrus/scvelo_umap_embedding.npy",
    },
    "larry": {
        "X": "./data/real_data_benchmark/larry/X_raw.npy",
        "V": "./data/real_data_benchmark/larry/V_raw.npy",
        "color": "./data/real_data_benchmark/larry/distance_pseudotime.npy",
        "embedding": "./data/real_data_benchmark/larry/flowmap_embedding.npy",
    },
}

In [ ]:
# ============================================================
#  DYNAMO FULL PROCEDURE BENCHMARK
# ============================================================

import numpy as np
import anndata
import dynamo as dyn
import matplotlib.pyplot as plt

# -----------------------------
# SELECT DATASET
# -----------------------------
dataset_name = "cell_cycle"

paths = DATASETS[dataset_name]

# -----------------------------
# LOAD DATA
# -----------------------------
X = np.load(paths["X"])
V = np.load(paths["V"])
C = np.load(paths["color"])

print(f"\nLoaded dataset: {dataset_name}")
print(f"X shape: {X.shape}, V shape: {V.shape}")

# -----------------------------
# BUILD ANNDATA
# -----------------------------
adata = anndata.AnnData(X)

adata.var_names = [
    f"dim{j}" for j in range(X.shape[1])
]

adata.obs["color"] = np.asarray(C, dtype=float)

adata.layers["X_raw"] = X
adata.layers["V_raw"] = V

# -----------------------------
# NEIGHBOR GRAPH
# -----------------------------
dyn.tl.neighbors(
    adata,
    X_data=X,
    n_neighbors=30,
    basis=None,
    layer=None,
)

# -----------------------------
# DYNAMO DIMENSION REDUCTION
# -----------------------------
dyn.tl.reduceDimension(
    adata,

    X_data=X,

    n_pca_components=30,

    n_neighbors=30,

    reduction_method="umap",

    min_dist=0.6,

    enforce=True,
)

# -----------------------------
# DYNAMO VELOCITY PROJECTION
# -----------------------------
dyn.tl.cell_velocities(
    adata,

    ekey="X_raw",
    vkey="V_raw",

    X=X,
    V=V,

    X_embedding=adata.obsm["X_umap"],

    basis="umap",

    transition_genes=list(adata.var_names),

    method="pearson",

    enforce=True,
)

print("Velocity projection complete.")

# -----------------------------
# EPSILON JITTER
# -----------------------------
np.random.seed(0)

eps = 1e-6

adata.obsm["X_umap"] += eps * np.random.randn(
    *adata.obsm["X_umap"].shape
)

adata.obsm["velocity_umap"] += eps * np.random.randn(
    *adata.obsm["velocity_umap"].shape
)

# -----------------------------
# ROBUST AXIS LIMITS
# -----------------------------
emb = adata.obsm["X_umap"]

x = emb[:, 0]
y = emb[:, 1]

xpad = 0.03 * (
    np.percentile(x, 99) - np.percentile(x, 1)
)

ypad = 0.03 * (
    np.percentile(y, 99) - np.percentile(y, 1)
)

xmin = np.percentile(x, 1) - xpad
xmax = np.percentile(x, 99) + xpad

ymin = np.percentile(y, 1) - ypad
ymax = np.percentile(y, 99) + ypad

# -----------------------------
# PLOT
# -----------------------------
fig, ax = plt.subplots(figsize=(6, 5))

dyn.pl.streamline_plot(
    adata,

    basis="umap",

    color="color",

    pointsize=0.5,

    density=0.3,

    linewidth=2.5,

    arrowsize=2.5,

    streamline_alpha=1.0,

    xy_grid_nums=[50, 50],

    show_legend=False,

    show_arrowed_spines=False,

    ax=ax,

    save_show_or_return="return",
)

# ax.set_xlim(xmin, xmax)
# ax.set_ylim(ymin, ymax)

ax.set_aspect("equal")

ax.set_xticks([])
ax.set_yticks([])

for spine in ax.spines.values():
    spine.set_visible(False)

ax.set_title("")

plt.tight_layout(pad=0.1)


save_path = f"./figures/benchmark/{dataset_name}_dynamo_streamline.png"

plt.savefig(
    save_path,
    dpi=600,
    bbox_inches="tight",
    pad_inches=0,
    transparent=True,
)

print(f"Saved to: {save_path}")

plt.show()

In [ ]:
# ============================================================
#  DYNAMO FULL PROCEDURE BENCHMARK
#  External X/V + Native Dynamo UMAP
# ============================================================

import numpy as np
import anndata
import dynamo as dyn
import matplotlib.pyplot as plt

# -------------------------------------------------
# DATASET
# -------------------------------------------------
dataset_name = "larry"

paths = DATASETS[dataset_name]

# -------------------------------------------------
# LOAD DATA
# -------------------------------------------------
X = np.load(paths["X"])
V = np.load(paths["V"])
C = np.load(paths["color"])

print(f"\nLoaded dataset: {dataset_name}")
print(f"X shape: {X.shape}")
print(f"V shape: {V.shape}")

# -------------------------------------------------
# CLEAN COLOR / PSEUDOTIME
# -------------------------------------------------
C = np.asarray(C, dtype=float).ravel()

C = np.nan_to_num(
    C,
    nan=0.0,
    posinf=0.0,
    neginf=0.0,
)

# clip extreme upper tail only
q95 = np.quantile(C, 0.95)
C = np.clip(C, None, q95)

# -------------------------------------------------
# BUILD ANNDATA
# -------------------------------------------------
adata = anndata.AnnData(X)

adata.var_names = [
    f"gene_{i}" for i in range(X.shape[1])
]

adata.obs["color"] = C

# IMPORTANT:
# Dynamo expects canonical naming
adata.layers["X"] = X.copy()
adata.layers["velocity"] = V.copy()

# also set main matrix
adata.X = X.copy()

# -------------------------------------------------
# DYNAMO PCA + UMAP
# -------------------------------------------------
dyn.tl.reduceDimension(
    adata, min_dist=0.5
)

print("UMAP complete.")

# -------------------------------------------------
# MOMENTS
# -------------------------------------------------
dyn.tl.moments(
    adata,

    X_data=X,

    n_pca_components=30,

    n_neighbors=30,
)

print("Moments complete.")

# -------------------------------------------------
# VELOCITY PROJECTION
# -------------------------------------------------
dyn.tl.cell_velocities(
    adata,

    ekey="X",
    vkey="velocity",

    X=X,
    V=V,

    X_embedding=adata.obsm["X_umap"],

    basis="umap",

    method="pearson",

    enforce=True,
)

print("Velocity projection complete.")

# -------------------------------------------------
# EPSILON JITTER
# -------------------------------------------------
np.random.seed(0)

eps = 1e-6

adata.obsm["X_umap"] += eps * np.random.randn(
    *adata.obsm["X_umap"].shape
)

adata.obsm["velocity_umap"] += eps * np.random.randn(
    *adata.obsm["velocity_umap"].shape
)

# -------------------------------------------------
# ROBUST AXIS LIMITS
# -------------------------------------------------
emb = adata.obsm["X_umap"]

x = emb[:, 0]
y = emb[:, 1]

xpad = 0.03 * (
    np.percentile(x, 99) - np.percentile(x, 1)
)

ypad = 0.03 * (
    np.percentile(y, 99) - np.percentile(y, 1)
)

xmin = np.percentile(x, 1) - xpad
xmax = np.percentile(x, 99) + xpad

ymin = np.percentile(y, 1) - ypad
ymax = np.percentile(y, 99) + ypad

# -------------------------------------------------
# PLOT
# -------------------------------------------------
fig, ax = plt.subplots(figsize=(6, 5))

dyn.pl.streamline_plot(
    adata,

    basis="umap",

    color="color",

    # points
    pointsize=0.2,
    alpha=0.08,

    # streamlines
    density=0.35,
    linewidth=1.8,
    arrowsize=2.0,
    streamline_alpha=1.0,

    # grid
    xy_grid_nums=[50, 50],

    # cosmetics
    show_legend=False,
    show_arrowed_spines=False,

    ax=ax,

    save_show_or_return="return",
)

# -------------------------------------------------
# CLEAN AXES
# -------------------------------------------------
ax.set_xlim(xmin, xmax)
ax.set_ylim(ymin, ymax)

ax.set_aspect("equal")

ax.set_xticks([])
ax.set_yticks([])

for spine in ax.spines.values():
    spine.set_visible(False)

ax.set_title("")

plt.tight_layout(pad=0.1)

plt.show()

In [ ]:
# -------------------------------------------------
# STREAMLINE PLOT
# -------------------------------------------------
fig, ax = plt.subplots(figsize=(6, 5))

dyn.pl.streamline_plot(
    adata,
    basis="umap",
    color="color",          # <-- true pseudotime
    pointsize=0.1,
    alpha=0.05,
    ax=ax,
    density=0.4,
    arrowsize=2.0,
    linewidth=1.5,
    streamline_alpha=1.0,
    xy_grid_nums=[50, 50],
    show_legend=False,
    show_arrowed_spines=False,
    save_show_or_return="return",
)

ax.set_title(dataset_name)
ax.set_aspect("equal")
ax.axis("off")

save_path = f"./figures/benchmark/{dataset_name}_dynamo_streamline.png"

plt.savefig(
    save_path,
    dpi=600,
    bbox_inches="tight",
    pad_inches=0,
    transparent=True,
)
plt.show()

In [ ]:
adata